# 임베딩 모델 4종 비교 — BGE-m3 / Qwen3-Embedding / KoSimCSE / KoE5

한국 KOSIS 통계표 매칭(3단계) 파이프라인에서 지금 쓰는 더미(해시 기반) 임베딩을 실제 모델로 대체하기 위한 비교 실험입니다.

**평가 방법**: `agent/mapping/table_catalog.json`의 표 7개(embedding_text)를 문서로, `agent/mapping/test_mapping.py`의 라벨링된 claim 문장 12개를 쿼리로 삼아 코사인 유사도 top-1 정확도를 비교합니다.

**비교 대상 4개 모델:**
1. **BGE-m3** (`BAAI/bge-m3`, ~568M) — 다국어 범용
2. **Qwen3-Embedding-4B** (`Qwen/Qwen3-Embedding-4B`) — 다국어 범용, 2025년 최신
3. **KoSimCSE** (`BM-K/KoSimCSE-roberta-multitask`, ~110M) — 한국어 전용, RoBERTa 기반
4. **KoE5** (`nlpai-lab/KoE5`) — 한국어 전용, E5 파인튜닝

4개 모두 서로 버전 충돌이 없어서(예전에 시도했던 NV-Embed-v2만 구형 `transformers`가 필요해서 문제였음), **런타임 재시작 없이 위에서부터 순서대로 한 번에 실행**하면 됩니다.

**준비물**: 런타임 > 런타임 유형 변경 > GPU(T4 이상) 선택.

## 0. 공통 설치 및 데이터 준비

In [6]:
!pip install -q -U sentence-transformers transformers accelerate FlagEmbedding
!pip install -U transformers


In [7]:
import time
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("⚠️ GPU가 안 잡혔습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요.")

import transformers
print("transformers 버전:", transformers.__version__)


device: cuda
transformers 버전: 5.14.1


In [8]:
import json
import os

# KOSIS 실제 통계표 1053개 (7개 정답 표 + 6개 주제(인구/노동/물가/국민계정/무역/주거)에서
# 실제 크롤링한 대규모 후보군). 7개짜리 큐레이션 카탈로그만으로는 변별력이 없어서
# (비슷한 표가 없어 아무 모델이나 100% 맞힘), 진짜 KOSIS 트리를 크롤링해 헷갈리는
# 유사 표들(예: 연령별/교육정도별/행정구역별 경제활동인구 등)을 대량으로 섞어 넣음.
CORPUS_PATH = "kosis_large_corpus.json"
if not os.path.exists(CORPUS_PATH):
    raise FileNotFoundError(
        f"'{CORPUS_PATH}'가 없습니다. 이 노트북과 같은 폴더의 kosis_large_corpus.json 파일을 "
        "Colab 왼쪽 파일 탭에 업로드하세요 (파일 아이콘 > 업로드)."
    )
with open(CORPUS_PATH, encoding="utf-8") as f:
    TABLES = json.load(f)

# agent/mapping/test_mapping.py의 원래 12개 + 변별력 강화를 위해 38개 추가한 50개 테스트 케이스
# (문장, claim_type, 정답 table_id, 카테고리)
# 각 카테고리마다: 키워드가 그대로 겹치는 "쉬운" 문장 + 의역/간접 표현 + 일부러 헷갈리게 만든 "애매한" 문장을 섞음
TEST_CASES = [
    # 고용/노동 (DT_1DA7001S) - 8개
    ("지난달 청년 실업률이 6%에 육박했다", "규모", "DT_1DA7001S", "고용/노동"),
    ("고용률이 역대 최고치를 기록했다", "비교", "DT_1DA7001S", "고용/노동"),
    ("취업자 수가 46개월 만에 감소 전환했다", "증감률", "DT_1DA7001S", "고용/노동"),
    ("일할 사람을 구하는 이들보다 놀고 있는 사람이 더 늘었다", "비교", "DT_1DA7001S", "고용/노동"),
    ("직장을 가진 국민의 비율이 소폭 상승했다", "증감률", "DT_1DA7001S", "고용/노동"),
    ("청년들이 일자리를 찾지 못해 거리로 나섰다", "규모", "DT_1DA7001S", "고용/노동"),
    ("경기 둔화로 기업들이 채용을 줄이면서 시장이 얼어붙었다", "비교", "DT_1DA7001S", "고용/노동"),
    ("은퇴 세대가 늘며 경제활동 참가율이 낮아졌다", "증감률", "DT_1DA7001S", "고용/노동"),

    # 물가/CPI (DT_1J22003) - 7개
    ("지난달 소비자물가가 전년 동월 대비 2.2% 올랐다", "증감률", "DT_1J22003", "물가/CPI"),
    ("생활물가가 5개월 연속 올랐다", "증감률", "DT_1J22003", "물가/CPI"),
    ("장바구니 물가가 서민 가계를 압박하고 있다", "규모", "DT_1J22003", "물가/CPI"),
    ("마트에서 장 보는 비용이 눈에 띄게 늘었다", "증감률", "DT_1J22003", "물가/CPI"),
    ("밥상 물가 부담이 커지며 소비 심리가 위축됐다", "규모", "DT_1J22003", "물가/CPI"),
    ("라면, 과자 등 가공식품 가격이 줄줄이 인상됐다", "규모", "DT_1J22003", "물가/CPI"),
    ("치솟는 물가에 서민들의 살림살이가 팍팍해졌다", "규모", "DT_1J22003", "물가/CPI"),

    # 인구 (DT_1B04005N) - 7개
    ("전국 주민등록인구가 5000만명 아래로 떨어졌다", "규모", "DT_1B04005N", "인구"),
    ("인구 감소가 가입자 이탈에 영향을 줬다", "비교", "DT_1B04005N", "인구"),
    ("우리나라에 사는 사람 수가 계속 줄어들고 있다", "비교", "DT_1B04005N", "인구"),
    ("지방 소멸 위기 속에 지역 인구가 급격히 줄고 있다", "비교", "DT_1B04005N", "인구"),
    ("빈집이 늘고 학교가 문을 닫는 등 인구 감소의 그림자가 짙어졌다", "전망", "DT_1B04005N", "인구"),
    ("전국 총인구가 사상 처음으로 감소세로 돌아섰다", "역대기록", "DT_1B04005N", "인구"),
    ("인구 순유출이 이어지며 지역 소멸 우려가 커지고 있다", "전망", "DT_1B04005N", "인구"),

    # 경제성장 (DT_200Y102) - 7개
    ("한국 경제성장률이 3개 분기 연속 0%대에 머물렀다", "비교", "DT_200Y102", "경제성장"),
    ("올해 경제성장률 전망치가 1.8%로 하향 조정됐다", "전망", "DT_200Y102", "경제성장"),
    ("나라 경제 전체 규모가 지난 분기보다 소폭 커졌다", "증감률", "DT_200Y102", "경제성장"),
    ("국내 경제가 눈에 띄게 활력을 잃어가고 있다", "비교", "DT_200Y102", "경제성장"),
    ("한국은행이 올해 성장 경로가 당초 예상보다 완만할 것이라고 밝혔다", "전망", "DT_200Y102", "경제성장"),
    ("내수 부진과 수출 둔화가 겹치며 경제 전반이 흔들렸다", "비교", "DT_200Y102", "경제성장"),
    ("우리 경제의 잠재성장률이 계속 낮아지고 있다는 우려가 나온다", "전망", "DT_200Y102", "경제성장"),

    # 무역/수출입 (DT_1R11006_FRM101) - 7개
    ("지난해 수출이 6838억달러로 역대 최대를 기록했다", "규모", "DT_1R11006_FRM101", "무역/수출입"),
    ("무역수지가 3년 만에 흑자로 전환했다", "비교", "DT_1R11006_FRM101", "무역/수출입"),
    ("해외로 내다 판 물건 값어치가 사상 최고를 찍었다", "역대기록", "DT_1R11006_FRM101", "무역/수출입"),
    ("반도체 훈풍에 힘입어 해외 판매가 늘었다", "증감률", "DT_1R11006_FRM101", "무역/수출입"),
    ("미국발 관세 여파로 대외 교역 여건이 악화됐다", "전망", "DT_1R11006_FRM101", "무역/수출입"),
    ("글로벌 경기 둔화가 국내 제조업 수출 전선에 그림자를 드리웠다", "전망", "DT_1R11006_FRM101", "무역/수출입"),
    ("대중국 수출이 부진하며 무역수지에 부담을 줬다", "비교", "DT_1R11006_FRM101", "무역/수출입"),

    # 부동산/주택 (DT_30404_B012) - 7개
    ("전국 집값이 하락세를 보였다", "비교", "DT_30404_B012", "부동산/주택"),
    ("서울 아파트값이 다시 오름세로 돌아섰다", "비교", "DT_30404_B012", "부동산/주택"),
    ("내 집 마련 비용 부담이 갈수록 커지고 있다", "증감률", "DT_30404_B012", "부동산/주택"),
    ("부동산 시장이 얼어붙으며 거래가 뚝 끊겼다", "규모", "DT_30404_B012", "부동산/주택"),
    ("강남 3구를 중심으로 매물이 자취를 감췄다", "규모", "DT_30404_B012", "부동산/주택"),
    ("고금리 장기화로 주택 수요가 크게 위축됐다", "비교", "DT_30404_B012", "부동산/주택"),
    ("전셋값도 동반 상승하며 주거비 부담이 커졌다", "증감률", "DT_30404_B012", "부동산/주택"),

    # 출생/사망/혼인 (DT_1B8000G) - 7개
    ("혼인 건수가 역대 최저를 기록했다", "역대기록", "DT_1B8000G", "출생/사망/혼인"),
    ("합계출산율이 0.7명대로 떨어졌다", "규모", "DT_1B8000G", "출생/사망/혼인"),
    ("아이 낳는 사람이 갈수록 줄어들고 있다", "비교", "DT_1B8000G", "출생/사망/혼인"),
    ("결혼하는 커플이 눈에 띄게 감소했다", "비교", "DT_1B8000G", "출생/사망/혼인"),
    ("저출생 고령화 여파로 학령인구가 급감했다", "비교", "DT_1B8000G", "출생/사망/혼인"),
    ("비혼과 만혼이 확산되며 가족 형태가 달라지고 있다", "전망", "DT_1B8000G", "출생/사망/혼인"),
    ("인구 절벽이라는 말이 나올 정도로 신생아 수가 급감했다", "비교", "DT_1B8000G", "출생/사망/혼인"),
]

doc_texts = [t["embedding_text"] for t in TABLES]
query_texts = [c[0] for c in TEST_CASES]
print(f"표 {len(TABLES)}개, 테스트 케이스 {len(TEST_CASES)}개 로드 완료")


표 1053개, 테스트 케이스 50개 로드 완료


In [9]:
def cosine_sim_matrix(query_vecs: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    """(n_query, dim) x (n_doc, dim) -> (n_query, n_doc) 코사인 유사도"""
    q = query_vecs / (np.linalg.norm(query_vecs, axis=1, keepdims=True) + 1e-12)
    d = doc_vecs / (np.linalg.norm(doc_vecs, axis=1, keepdims=True) + 1e-12)
    return q @ d.T


def evaluate(model_name: str, doc_vecs: np.ndarray, query_vecs: np.ndarray, k_list=(1, 5, 10)) -> dict:
    """top-1 정확도 + Recall@k + 1등-2등 점수 마진까지 계산. doc_vecs 순서는 TABLES와 동일해야 함.

    - Recall@k: 정답이 상위 k개 안에만 들어오면 맞은 것으로 침 (top-1을 못 맞혀도 재랭커가
      보완할 여지가 있는지 보는 지표 - 우리 실제 파이프라인은 임베딩 top-k를 keyword_search와
      합쳐서 재정렬하므로, top-1보다 Recall@5/10이 실제 쓰임새에 더 가까움).
    - 마진: 1등과 2등의 코사인 유사도 차이. 마진이 크면 그 판단이 안정적이라는 뜻이고,
      마진이 작으면 정답을 맞혔더라도 문장이 조금만 바뀌어도 순위가 뒤집힐 수 있는
      불안정한 판단이라는 뜻.
    """
    doc_vecs = np.asarray(doc_vecs)
    query_vecs = np.asarray(query_vecs)
    sims = cosine_sim_matrix(query_vecs, doc_vecs)
    table_ids = [t["tblId"] for t in TABLES]

    correct_at_k = {k: 0 for k in k_list}
    margins = []
    rows = []

    for i, (sentence, claim_type, expected_id, category) in enumerate(TEST_CASES):
        order = np.argsort(-sims[i])  # 유사도 내림차순 문서 인덱스
        ranked_ids = [table_ids[idx] for idx in order]

        predicted_id = ranked_ids[0]
        ok = predicted_id == expected_id

        for k in k_list:
            if expected_id in ranked_ids[:k]:
                correct_at_k[k] += 1

        margin = float(sims[i][order[0]] - sims[i][order[1]])
        margins.append(margin)

        rank_of_correct = ranked_ids.index(expected_id) + 1 if expected_id in ranked_ids else None
        rows.append((category, sentence, expected_id, predicted_id, float(sims[i][order[0]]), ok, rank_of_correct))

    total = len(TEST_CASES)
    mean_margin = float(np.mean(margins))

    print()
    print(f"=== {model_name} ===")
    for k in k_list:
        print(f"  Recall@{k}: {correct_at_k[k]}/{total} ({correct_at_k[k] / total * 100:.1f}%)")
    print(f"  평균 1등-2등 마진: {mean_margin:.4f}")
    print()
    for category, sentence, expected_id, predicted_id, score, ok, rank_of_correct in rows:
        mark = "O" if ok else "X"
        print(f"[{mark}] [{category}] {sentence}")
        print(f"      기대={expected_id} 예측={predicted_id} score={score:.3f} 정답순위={rank_of_correct}")

    return {
        "model": model_name,
        "correct": correct_at_k[1],
        "total": total,
        "accuracy": correct_at_k[1] / total,
        "recall_at_5": correct_at_k.get(5, correct_at_k[1]) / total,
        "recall_at_10": correct_at_k.get(10, correct_at_k[1]) / total,
        "mean_margin": mean_margin,
    }


RESULTS = {}  # 모델명 -> evaluate() 반환값. 아래 섹션들을 실행할 때마다 채워짐.


---
## 1. BGE-m3 (BAAI/bge-m3, ~568M)

다국어 지원, dense/sparse/multi-vector 다 지원하지만 여기선 dense 벡터만 씁니다.

In [10]:
from FlagEmbedding import BGEM3FlagModel

t0 = time.time()
bge_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(device == "cuda"))

doc_vecs = bge_model.encode(doc_texts)["dense_vecs"]
query_vecs = bge_model.encode(query_texts)["dense_vecs"]
print(f"임베딩 완료 ({time.time() - t0:.1f}s), dim={np.array(doc_vecs).shape[1]}")

RESULTS["BGE-m3"] = evaluate("BGE-m3", doc_vecs, query_vecs)

del bge_model
torch.cuda.empty_cache()


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:00<00:00, 18.29it/s]


임베딩 완료 (5.2s), dim=1024

=== BGE-m3: top-1 정확도 5/50 ===
[X] [고용/노동] 지난달 청년 실업률이 6%에 육박했다
      기대=DT_1DA7001S 예측=DT_1DA7105S score=0.581
[X] [고용/노동] 고용률이 역대 최고치를 기록했다
      기대=DT_1DA7001S 예측=DT_1DA7E08S_NEW score=0.577
[X] [고용/노동] 취업자 수가 46개월 만에 감소 전환했다
      기대=DT_1DA7001S 예측=DT_1DE7081S score=0.529
[X] [고용/노동] 일할 사람을 구하는 이들보다 놀고 있는 사람이 더 늘었다
      기대=DT_1DA7001S 예측=DT_1DA7E08S_NEW score=0.555
[X] [고용/노동] 직장을 가진 국민의 비율이 소폭 상승했다
      기대=DT_1DA7001S 예측=DT_1DE7081S score=0.639
[X] [고용/노동] 청년들이 일자리를 찾지 못해 거리로 나섰다
      기대=DT_1DA7001S 예측=DT_1DA7090S score=0.548
[X] [고용/노동] 경기 둔화로 기업들이 채용을 줄이면서 시장이 얼어붙었다
      기대=DT_1DA7001S 예측=DT_1DA7E06S_NEW score=0.479
[X] [고용/노동] 은퇴 세대가 늘며 경제활동 참가율이 낮아졌다
      기대=DT_1DA7001S 예측=DT_1DE9046S score=0.670
[X] [물가/CPI] 지난달 소비자물가가 전년 동월 대비 2.2% 올랐다
      기대=DT_1J22003 예측=DT_1J22042 score=0.649
[X] [물가/CPI] 생활물가가 5개월 연속 올랐다
      기대=DT_1J22003 예측=DT_1J22042 score=0.598
[O] [물가/CPI] 장바구니 물가가 서민 가계를 압박하고 있다
      기대=DT_1J22003 예측=DT_1J22003 score=0.555
[X] [물가/

---
## 2. Qwen3-Embedding-4B (Qwen/Qwen3-Embedding-4B)

쿼리 쪽에 instruction(task 설명)을 붙이는 걸 권장합니다. 문서(테이블 설명) 쪽은 instruction 없이 그대로 인코딩합니다.
VRAM이 부족하면 모델명을 `Qwen/Qwen3-Embedding-0.6B`로 바꿔서 재실행하세요.

In [11]:
from sentence_transformers import SentenceTransformer

QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # VRAM 부족하면 "Qwen/Qwen3-Embedding-0.6B"로 교체

t0 = time.time()
qwen_model = SentenceTransformer(
    QWEN_MODEL_NAME,
    model_kwargs={"torch_dtype": torch.float16} if device == "cuda" else {},
    device=device,
)

instruction = "Given a Korean news claim sentence, retrieve the KOSIS statistical table description that best matches it"
doc_vecs = qwen_model.encode(doc_texts, convert_to_numpy=True, show_progress_bar=True)
query_vecs = qwen_model.encode(
    [f"Instruct: {instruction}\nQuery: {q}" for q in query_texts],
    convert_to_numpy=True, show_progress_bar=True,
)
print(f"임베딩 완료 ({time.time() - t0:.1f}s), dim={doc_vecs.shape[1]}")

RESULTS["Qwen3-Embedding"] = evaluate("Qwen3-Embedding", doc_vecs, query_vecs)

del qwen_model
torch.cuda.empty_cache()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

임베딩 완료 (39.3s), dim=2560

=== Qwen3-Embedding: top-1 정확도 18/50 ===
[X] [고용/노동] 지난달 청년 실업률이 6%에 육박했다
      기대=DT_1DA7001S 예측=DT_1DA7102S score=0.624
[O] [고용/노동] 고용률이 역대 최고치를 기록했다
      기대=DT_1DA7001S 예측=DT_1DA7001S score=0.590
[X] [고용/노동] 취업자 수가 46개월 만에 감소 전환했다
      기대=DT_1DA7001S 예측=DT_1DA9003S score=0.598
[X] [고용/노동] 일할 사람을 구하는 이들보다 놀고 있는 사람이 더 늘었다
      기대=DT_1DA7001S 예측=DT_1DE9046S score=0.625
[X] [고용/노동] 직장을 가진 국민의 비율이 소폭 상승했다
      기대=DT_1DA7001S 예측=DT_1DE9046S score=0.624
[X] [고용/노동] 청년들이 일자리를 찾지 못해 거리로 나섰다
      기대=DT_1DA7001S 예측=DT_1DA7102S score=0.610
[X] [고용/노동] 경기 둔화로 기업들이 채용을 줄이면서 시장이 얼어붙었다
      기대=DT_1DA7001S 예측=DT_1DA7E06S_NEW score=0.521
[X] [고용/노동] 은퇴 세대가 늘며 경제활동 참가율이 낮아졌다
      기대=DT_1DA7001S 예측=DT_1DE9046S score=0.661
[X] [물가/CPI] 지난달 소비자물가가 전년 동월 대비 2.2% 올랐다
      기대=DT_1J22003 예측=DT_1J22042 score=0.607
[X] [물가/CPI] 생활물가가 5개월 연속 올랐다
      기대=DT_1J22003 예측=DT_1J22042 score=0.596
[O] [물가/CPI] 장바구니 물가가 서민 가계를 압박하고 있다
      기대=DT_1J22003 예측=DT_1J22003 score=0.552
[X] [

---
## 3. KoSimCSE (BM-K/KoSimCSE-roberta-multitask, ~110M)

한국어 전용 문장임베딩 모델. sentence-transformers 형식이 아니라 `AutoModel` + 수동 풀링 방식으로 씁니다 — 관례상 **CLS 토큰**(마지막 hidden state의 첫 번째 토큰)을 문장 임베딩으로 사용합니다.

In [12]:
from transformers import AutoModel, AutoTokenizer

KOSIMCSE_MODEL_NAME = "BM-K/KoSimCSE-roberta-multitask"

t0 = time.time()
kosimcse_model = AutoModel.from_pretrained(KOSIMCSE_MODEL_NAME).to(device)
kosimcse_model.eval()
kosimcse_tokenizer = AutoTokenizer.from_pretrained(KOSIMCSE_MODEL_NAME)


def embed_kosimcse(texts, batch_size=16):
    all_vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = kosimcse_tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
            outputs = kosimcse_model(**inputs, return_dict=True)
            cls_vec = outputs.last_hidden_state[:, 0]  # CLS 토큰 풀링 (KoSimCSE 관례)
            all_vecs.append(cls_vec.cpu().numpy())
    return np.concatenate(all_vecs, axis=0)


doc_vecs = embed_kosimcse(doc_texts)
query_vecs = embed_kosimcse(query_texts)
print(f"임베딩 완료 ({time.time() - t0:.1f}s), dim={doc_vecs.shape[1]}")

RESULTS["KoSimCSE"] = evaluate("KoSimCSE", doc_vecs, query_vecs)

del kosimcse_model
torch.cuda.empty_cache()


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

임베딩 완료 (8.4s), dim=768

=== KoSimCSE: top-1 정확도 3/50 ===
[X] [고용/노동] 지난달 청년 실업률이 6%에 육박했다
      기대=DT_1DA7001S 예측=DT_1DE9048S score=0.629
[X] [고용/노동] 고용률이 역대 최고치를 기록했다
      기대=DT_1DA7001S 예측=DT_1DA7E43S_NEW score=0.621
[X] [고용/노동] 취업자 수가 46개월 만에 감소 전환했다
      기대=DT_1DA7001S 예측=DT_1DA7089S score=0.583
[X] [고용/노동] 일할 사람을 구하는 이들보다 놀고 있는 사람이 더 늘었다
      기대=DT_1DA7001S 예측=DT_1DA7A64S score=0.490
[X] [고용/노동] 직장을 가진 국민의 비율이 소폭 상승했다
      기대=DT_1DA7001S 예측=DT_1DA7A64S score=0.701
[X] [고용/노동] 청년들이 일자리를 찾지 못해 거리로 나섰다
      기대=DT_1DA7001S 예측=DT_1DA7086S score=0.479
[X] [고용/노동] 경기 둔화로 기업들이 채용을 줄이면서 시장이 얼어붙었다
      기대=DT_1DA7001S 예측=DT_1DA9001S score=0.503
[X] [고용/노동] 은퇴 세대가 늘며 경제활동 참가율이 낮아졌다
      기대=DT_1DA7001S 예측=DT_1DA7179S score=0.619
[X] [물가/CPI] 지난달 소비자물가가 전년 동월 대비 2.2% 올랐다
      기대=DT_1J22003 예측=DT_1J22001 score=0.605
[X] [물가/CPI] 생활물가가 5개월 연속 올랐다
      기대=DT_1J22003 예측=DT_1J22042 score=0.554
[X] [물가/CPI] 장바구니 물가가 서민 가계를 압박하고 있다
      기대=DT_1J22003 예측=DT_1J22041 score=0.517
[X] [물가/CPI] 마트

---
## 4. KoE5 (nlpai-lab/KoE5)

한국어 전용으로 파인튜닝된 E5 계열 모델(고려대 NLP&AI Lab). E5 계열 관례상 쿼리에는 `"query: "`, 문서에는 `"passage: "` 접두어를 붙입니다.

In [13]:
t0 = time.time()
koe5_model = SentenceTransformer("nlpai-lab/KoE5", device=device)

doc_vecs = koe5_model.encode([f"passage: {t}" for t in doc_texts], convert_to_numpy=True, show_progress_bar=True)
query_vecs = koe5_model.encode([f"query: {q}" for q in query_texts], convert_to_numpy=True, show_progress_bar=True)
print(f"임베딩 완료 ({time.time() - t0:.1f}s), dim={doc_vecs.shape[1]}")

RESULTS["KoE5"] = evaluate("KoE5", doc_vecs, query_vecs)

del koe5_model
torch.cuda.empty_cache()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

임베딩 완료 (21.0s), dim=1024

=== KoE5: top-1 정확도 0/50 ===
[X] [고용/노동] 지난달 청년 실업률이 6%에 육박했다
      기대=DT_1DA7001S 예측=DT_1DA7105S score=0.536
[X] [고용/노동] 고용률이 역대 최고치를 기록했다
      기대=DT_1DA7001S 예측=DT_1DA7A64S score=0.417
[X] [고용/노동] 취업자 수가 46개월 만에 감소 전환했다
      기대=DT_1DA7001S 예측=DT_1DA7C06S score=0.532
[X] [고용/노동] 일할 사람을 구하는 이들보다 놀고 있는 사람이 더 늘었다
      기대=DT_1DA7001S 예측=DT_1D07012S score=0.371
[X] [고용/노동] 직장을 가진 국민의 비율이 소폭 상승했다
      기대=DT_1DA7001S 예측=DT_1DE7081S score=0.575
[X] [고용/노동] 청년들이 일자리를 찾지 못해 거리로 나섰다
      기대=DT_1DA7001S 예측=DT_1PC2022 score=0.355
[X] [고용/노동] 경기 둔화로 기업들이 채용을 줄이면서 시장이 얼어붙었다
      기대=DT_1DA7001S 예측=DT_1DE7070S score=0.362
[X] [고용/노동] 은퇴 세대가 늘며 경제활동 참가율이 낮아졌다
      기대=DT_1DA7001S 예측=DT_1DE9046S score=0.579
[X] [물가/CPI] 지난달 소비자물가가 전년 동월 대비 2.2% 올랐다
      기대=DT_1J22003 예측=DT_1J22042 score=0.633
[X] [물가/CPI] 생활물가가 5개월 연속 올랐다
      기대=DT_1J22003 예측=DT_1J22005 score=0.590
[X] [물가/CPI] 장바구니 물가가 서민 가계를 압박하고 있다
      기대=DT_1J22003 예측=DT_1J22005 score=0.424
[X] [물가/CPI] 마트에서 장 보는

---
## 5. 최종 비교

In [14]:
import pandas as pd

summary = pd.DataFrame([
    {
        "모델": name,
        "정답": r["correct"],
        "전체": r["total"],
        "top-1 정확도": f"{r['accuracy'] * 100:.1f}%",
        "Recall@5": f"{r['recall_at_5'] * 100:.1f}%",
        "Recall@10": f"{r['recall_at_10'] * 100:.1f}%",
        "평균 마진(1등-2등)": f"{r['mean_margin']:.3f}",
    }
    for name, r in RESULTS.items()
])
summary


,모델,정답,전체,정확도
0,BGE-m3,5,50,10.0%
1,Qwen3-Embedding,18,50,36.0%
2,KoSimCSE,3,50,6.0%
3,KoE5,0,50,0.0%
